In [ ]:
"""
AgroAlert Ghana -- Extended MODIS LST Collection (2019-01-01 to 2023-12-31)
=============================================================================
Same logic as original 04_soil_moisture.ipynb (final cell), extended date range.
Source: MODIS/061/MOD11A2 (8-day composite LST)
MOD11A2 coverage is available from 2000 onward, so 2019 poses no data
availability issue -- only quota/runtime.
"""

import ee
import pandas as pd
import os

credentials = ee.ServiceAccountCredentials(
    email='agroalert-service@ee-ishmaelfc.iam.gserviceaccount.com',
    key_file='C:/Users/ELITE/Documents/AGROALERT/service_account.json'
)
ee.Initialize(credentials)

communities = [
    {"name": "Tamale",           "region": "Northern",      "lat": 9.4008,  "lon": -0.8393},
    {"name": "Techiman",         "region": "Bono East",     "lat": 7.5833,  "lon": -1.9333},
    {"name": "Kumasi",           "region": "Ashanti",       "lat": 6.6885,  "lon": -1.6244},
    {"name": "Ho",               "region": "Volta",         "lat": 6.6000,  "lon":  0.4700},
    {"name": "Bolgatanga",       "region": "Upper East",    "lat": 10.7856, "lon": -0.8514},
    {"name": "Wa",               "region": "Upper West",    "lat": 10.0601, "lon": -2.5099},
    {"name": "Sunyani",          "region": "Bono",          "lat": 7.3349,  "lon": -2.3123},
    {"name": "Koforidua",        "region": "Eastern",       "lat": 6.0940,  "lon": -0.2591},
    {"name": "Cape Coast",       "region": "Central",       "lat": 5.1053,  "lon": -1.2466},
    {"name": "Sefwi Wiawso",     "region": "Western North", "lat": 6.2069,  "lon": -2.4856},
    {"name": "Damongo",          "region": "Savannah",      "lat": 9.0833,  "lon": -1.8167},
    {"name": "Nalerigu",         "region": "North East",    "lat": 10.5167, "lon": -0.3667},
    {"name": "Dambai",           "region": "Oti",           "lat": 8.0667,  "lon":  0.1833},
    {"name": "Goaso",            "region": "Ahafo",         "lat": 6.8017,  "lon": -2.5181},
    {"name": "Sekondi-Takoradi", "region": "Western",       "lat": 4.9347,  "lon": -1.7137},
]

START_DATE = '2019-01-01'
END_DATE = '2023-12-31'


def get_lst(community, start_date, end_date):
    point = ee.Geometry.Point([community['lon'], community['lat']])
    collection = (ee.ImageCollection('MODIS/061/MOD11A2')
        .filterBounds(point)
        .filterDate(start_date, end_date)
        .select('LST_Day_1km'))

    def extract(img):
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=point.buffer(10000),
            scale=1000
        )
        raw = val.get('LST_Day_1km')
        lst = ee.Algorithms.If(
            raw,
            ee.Number(raw).multiply(0.02).subtract(273.15),
            None
        )
        return ee.Feature(None, {
            'date': img.date().format('YYYY-MM-dd'),
            'lst_celsius': lst,
            'community': community['name'],
            'region': community['region']
        })

    return collection.map(extract)


print(f"Fetching MODIS LST for {len(communities)} communities, {START_DATE} to {END_DATE}")

all_records = []

# Pulling community-by-community for the same robustness reason as NDVI --
# 5 years of 8-day composites x 15 communities is a bigger payload than
# the original 2-year pull.
for i, community in enumerate(communities):
    print(f"  [{i+1}/{len(communities)}] Fetching {community['name']}...")
    fc = get_lst(community, START_DATE, END_DATE)
    try:
        data = fc.getInfo()
    except Exception as e:
        print(f"    ERROR fetching {community['name']}: {e}")
        continue

    count = 0
    for feature in data['features']:
        props = feature['properties']
        if props.get('lst_celsius') is not None:
            all_records.append({
                'community': props.get('community'),
                'region': props.get('region'),
                'date': props.get('date'),
                'lst_celsius': props.get('lst_celsius')
            })
            count += 1
    print(f"    -> {count} valid records")

df_lst = pd.DataFrame(all_records)
df_lst['date'] = pd.to_datetime(df_lst['date'])
df_lst = df_lst.sort_values(['community', 'date']).reset_index(drop=True)

os.makedirs('C:/Users/ELITE/Documents/AGROALERT/data_raw', exist_ok=True)
OUTPUT_PATH = 'C:/Users/ELITE/Documents/AGROALERT/data_raw/soil_moisture_raw_2019_2023.csv'
df_lst.to_csv(OUTPUT_PATH, index=False)

print(f"\nTotal LST records: {len(df_lst)}")
print(f"Saved to: {OUTPUT_PATH}")
print(f"Communities: {df_lst['community'].nunique()} / {len(communities)}")
print(f"Date range: {df_lst['date'].min()} to {df_lst['date'].max()}")

print("\n--- IMPORTANT ---")
print("This pull already spans the full 2019-2023 window, so replace (don't merge)")
print("your existing soil_moisture_raw.csv with this file to avoid duplicate rows.")
